EDA and Data cleaning of Titanic dataset (train one from Kaggle)

In [2]:
import numpy as np
import pandas as pd

In [3]:
#Load and look
df = pd.read_csv("train.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [4]:
#how many nulls per column
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [ ]:
#unique values
print(df['Pclass'].unique())

#unique value counts
print(df['Pclass'].value_counts())

[3 1 2]
Pclass
3    491
1    216
2    184
Name: count, dtype: int64


### Q3 Binning a feature (Fare)

In [6]:
#get quartiles
df['Fare'].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [ ]:
"""#define bin edges
bins = [0, 7.91, 14.45, 31, 520]

#labels
labels = ['Q1', 'Q2', 'Q3', 'Q4']

#cut to bin the data
df["Fare_bin"] = pd.cut(df["Fare"], bins, labels)

#group the new bin and find the range (min to max) of each band
range_by_bin = df.groupby("Fare_bin")["Fare"].agg(['min', 'max'])
print("\n Each band range:")
print(range_by_bin)"""


 Each band range:
                   min       max
Fare_bin                        
(0.0, 7.91]     4.0125    7.8958
(7.91, 14.45]   7.9250   14.4000
(14.45, 31.0]  14.4542   31.0000
(31.0, 520.0]  31.2750  512.3292


A better, more automatic way to do it

In [ ]:
#quartile boundaries using describe()
quarts = df["Fare"].quantile([0.25, .5, .75])
print("Quartile Boundaries:")
print(quarts)

#define bin edges
bins = [df["Fare"].min()] + list(quarts.values) + [df["Fare"].max()]

#labels
labels = ['Q1', 'Q2', 'Q3', 'Q4']

#cut to bin the data
#include_lowest to be inclusive
df["Fare_bin"] = pd.cut(df["Fare"], bins, labels, include_lowest=True)

#group the new bin and find the range (min to max) of each band
range_by_bin = df.groupby("Fare_bin")["Fare"].agg(['min', 'max'])
print("\n Each band range:")
print(range_by_bin)

Quartile Boundaries:
0.25     7.9104
0.50    14.4542
0.75    31.0000
Name: Fare, dtype: float64

 Each band range:
                     min       max
Fare_bin                          
(-0.001, 7.91]    0.0000    7.8958
(7.91, 14.454]    7.9250   14.4542
(14.454, 31.0]   14.4583   31.0000
(31.0, 512.329]  31.2750  512.3292


### Q4. What to do with the age column?

In [ ]:
#unique values
print(df['Age'].unique())

#unique value counts
print(df['Age'].value_counts())

[22.   38.   26.   35.     nan 54.    2.   27.   14.    4.   58.   20.
 39.   55.   31.   34.   15.   28.    8.   19.   40.   66.   42.   21.
 18.    3.    7.   49.   29.   65.   28.5   5.   11.   45.   17.   32.
 16.   25.    0.83 30.   33.   23.   24.   46.   59.   71.   37.   47.
 14.5  70.5  32.5  12.    9.   36.5  51.   55.5  40.5  44.    1.   61.
 56.   50.   36.   45.5  20.5  62.   41.   52.   63.   23.5   0.92 43.
 60.   10.   64.   13.   48.    0.75 53.   57.   80.   70.   24.5   6.
  0.67 30.5   0.42 34.5  74.  ]
Age
24.00    30
22.00    27
18.00    26
28.00    25
19.00    25
         ..
24.50     1
0.67      1
0.42      1
34.50     1
74.00     1
Name: count, Length: 88, dtype: int64


### Q5 Splitting Cabin column to deck and number

In [30]:
#first letter
df['Deck'] = df['Cabin'].str[0]
#rest of the column
df['Seat'] = df['Cabin'].str[1:]

#unique value counts
df['Deck'].value_counts(dropna=False)

Deck
NaN    687
C       59
B       47
D       33
E       32
A       15
F       13
G        4
T        1
Name: count, dtype: int64

In [14]:
df['Deck'] = df['Deck'].fillna('Unknown')

df['Deck'].value_counts()

Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

### Q7. Which deck has the best survival rate?

In [16]:
#calculate survival rate per deck
surv_rate = df.groupby('Deck')['Survived'].mean()

#find the deck with the highest survival rate
best_deck = surv_rate.idxmax()
best_rate = surv_rate.max()*100

print(f"Best Deck: {best_deck} with survival rate :{best_rate:.2f}%")

Best Deck: D with survival rate :75.76%


### Q8. Which pclasses were on decks A, B, and C

In [ ]:
#count each (Deck,Pclass) pair
pairs = df.value_counts(subset=['Deck', 'Pclass'])

#filter for decks A, B, C
print(pairs.loc[['A', 'B', 'C',]])

Deck  Pclass
A     1         15
B     1         47
C     1         59
D     1         29
      2          4
Name: count, dtype: int64


In [23]:
#out of personal curiosity I want to see the partition
#of classes on each deck

#a frequency table
partition = pd.crosstab(df['Deck'], df['Pclass'])

print(partition)

Pclass    1    2    3
Deck                 
A        15    0    0
B        47    0    0
C        59    0    0
D        29    4    0
E        25    4    3
F         0    8    5
G         0    0    4
T         1    0    0
Unknown  40  168  479


### Q6

In [27]:
print(df['Embarked'].value_counts(dropna=False))

print(df['Sex'].value_counts(dropna=False))

Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64
Sex
male      577
female    314
Name: count, dtype: int64
